# Somo 08 - Mtindo wa Muundo wa Wakala Wengi


## Mipangilio


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kwa Nini Mifumo ya Wakala Wengi?

Kazi za dunia halisi kama upangaji wa safari zinahusisha aina nyingi tofauti za ujuzi — usafirishaji, maarifa ya eneo, bajeti, na zaidi. Wakala mmoja akijaribu kushughulikia kila kitu haraka huwa mgumu kudhibiti.

Mifumo ya wakala wengi hutatua hili kupitia **utaalam**: kila wakala anazingatia eneo moja la ujuzi, akitoa matokeo bora zaidi kuliko mtaalamu wa jumla. Pia huboresha **uwezo wa kupanuka** — unaweza kuongeza mawakala wapya (kwa mfano, mtaalamu wa ndege, mkosoaji wa mikahawa) bila kuandika upya mtiririko wa kazi ulio tayari. Mawakala hugawanyika kupitia mchakato uliopangwa, wakipitisha muktadha kutoka kwa mmoja hadi mwingine.


## Kuunda Wakala Maalum


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Kuunda Mtiririko wa Kazi Mfululizo

`WorkflowBuilder` inakuwezesha kuunganisha mawakala katika mchoro ulioelekezwa. Hapa tunaunda mchakato rahisi wa hatua mbili: **TravelPlanner** huandaa ratiba, kisha **TravelConcierge** hupitia na kuiboresha.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Kuongeza Wakala Zaidi Kwenye Mtiririko wa Kazi

Moja ya faida kubwa za muundo wa wakala wengi ni jinsi ilivyo rahisi kuendeleza. Hapa chini tunaongeza wakala wa **BudgetReviewer** anayekagua mpango dhidi ya bajeti ya msafiri, kuonyesha vitu vinavyoweza kuongeza gharama kupita kikomo, na kupendekeza mbadala za kuokoa pesa. Mtiririko wa kazi sasa unafanya kazi za wakala watatu mfululizo:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Muhtasari

Katika somo hili ulijifunza jinsi ya:

1. **Kuunda mawakala maalum** — kila mmoja akiwa na jukumu mahususi (kipanga, hudumu, ukaguzi wa bajeti).
2. **Kuwaunganisha mawakala katika mchakato wa mfuatano** kwa kutumia `WorkflowBuilder` na `add_edge`.
3. **Kutiririsha matokeo** kutoka kwenye njia ya mawakala wengi, ukifuata ni wakala gani anazungumza.
4. **Kupanua mchakato** kwa kuongeza mawakala wapya kwenye mnyororo bila kubadilisha wale waliopo.

Mfumo wa muundo wa mawakala wengi unahakikisha kila wakala ni rahisi huku ukitoa matokeo yenye kina na mapitio ya kina zaidi kuliko mawakala mmoja mmoja angeweza kupata peke yake.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tangazo la Kuepuka Madhara**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Wakati tunajitahidi kufikia usahihi, tafadhali fahamu kuwa tafsiri za moja kwa moja zinaweza kuwa na makosa au upungufu wa usahihi. Hati asili katika lugha yake ya asili inapaswa kuzingatiwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu na ya mtu inashauriwa. Hatuwezi kuwajibika kwa kutoelewana au tafsiri potofu zinazotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
